# Unified model comparison

One metric set, computed identically for every model via `personal.shared_functions.unified_metrics`:

| metric | meaning |
|---|---|
| `ho_nse`  | **held-out** NSE = mean over strikes of \|model−mid\| / half-spread, prior (one-step-ahead) state |
| `is_nse`  | **in-sample** NSE, posterior (post-update) state |
| `ho_rmse` / `is_rmse` | same splits, RMSE in price units (C/df, dollars) |
| `neg_mass` | implied negative mass (Breeden–Litzenberger or native density); 0 = arbitrage-free |

All prices are undiscounted **call** prices C/df; puts are converted via put-call parity so every model is scored in the same space. Each model's adapter produces the *standard eval table* (`ts, strike, forward, mid_call, half_spread, prior_call, post_call`); `unified_metrics.evaluate` turns that into the metric row, and `comparison_table` stacks the rows.

In [ ]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (has pyproject.toml + quantdata/)."""
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists() and (d / "quantdata").is_dir():
            return d
    raise RuntimeError(f"quant_data project root not found above {start}")


PROJECT_ROOT = _find_root(Path.cwd().resolve())

# `quantdata` is an editable-installed package, so `import quantdata...` works
# already; PROJECT_ROOT is added so it also imports without the editable install.
# The `personal/` modules are flat scripts (no __init__.py), so every directory
# we import a module from must be on sys.path explicitly.
for p in (
    PROJECT_ROOT,
    PROJECT_ROOT / "personal" / "shared_functions",
    PROJECT_ROOT / "personal" / "SANOS_KF_Strike",
    PROJECT_ROOT / "personal" / "Kalman_filter_spline",
):
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import personal.SANOS_KF_Strike.sanos_ekf_eval as ekf

# personal/ flat modules:
import personal.shared_functions.unified_metrics as um

# quantdata package (example — import whatever you need):
import quantdata  # e.g. from quantdata.sanos import open as sanos_open
from personal.Kalman_filter_spline.surface_kalman_filters import build_iv_observations
from personal.SANOS_KF_Strike.sanos_kf_strike import (
    SanosKFStrikeConfig,
    gate_eval_table,
    run_sanos_kf_strike,
)
from quantdata.massive.client import MassiveClient

rows = []  # one metric dict per model, fed to um.comparison_table at the end

/Users/elie/Documents/GitHub/quant_data/quantdata/__init__.py:212: UserWarning: Using cache mode 'on' as specified in environment variable QUANTDATA_CACHE_MODE.


## CONFIG

In [2]:
TRADE_DATE = date(2024, 1, 3)

In [3]:
client = MassiveClient()

## 1. SANOS EKF (direct-observation, Option 2)

Runs from `phase1_cache_<date>.pkl`; snapshots are full smiles, so negative mass is computed per snapshot by Breeden–Litzenberger inside `evaluate`.

In [ ]:
# ekf.load_cache("2024-03-05")  # uncomment / change to switch dates
df_ekf = ekf.eval_table(q_scale=1.0, ell=3.0, p0_scale=1e-2, warm=False, correlated=True)
m_ekf = um.evaluate(df_ekf, model_name="SANOS EKF")
rows.append(m_ekf)
m_ekf

{'model': 'SANOS EKF',
 'ho_nse_mean': 0.4700187206951173,
 'ho_nse_med': 0.4237491808969624,
 'is_nse_mean': 0.19386728971496187,
 'is_nse_med': 0.1827296518597344,
 'ho_rmse_mean': 0.008731178524169384,
 'ho_rmse_med': 0.007141691476813369,
 'is_rmse_mean': 0.005225170429904656,
 'is_rmse_med': 0.003941555131580051,
 'neg_mass_mean': 0.0,
 'neg_mass_med': 0.0,
 'arb_frac': 0.0,
 'n_snapshots': 24066}

## 2. SANOS 2-stage (batch DLV fit → KF on DLV²)

The two-stage incumbent, ported from `sanos_dlv_kf_eval_lean.ipynb` with **its exact hyperparameters**:

- **Stage 1 — batch DLV fit.** At each `BATCH_FREQ` bucket, run the SANOS absolute-strike optimizer (`run_intraday_loop` / `fit_density`) to get a DLV² vector per snapshot.
- **Stage 2 — correlated-Q Kalman filter on DLV²** (`run_dlv_kalman_filter_v2`, `KFConfig`). Random-walk transition, cross-strike RBF process noise, selection-H for stale nodes.

`prior_call` is priced from the KF **prior** state `x_pred_hist[k]` (one-step-ahead, held-out); `post_call` from the KF **posterior** `x_hist[k]` (in-sample). Each step's DLV² is turned back into a density via the N×N exactly-determined linear system (`_density_from_kf_dlvsq`), priced with `option_prices`, then fed through `um.from_option_prices` / `um.evaluate`. Negative mass is computed per snapshot by Breeden–Litzenberger on the posterior curve.

Runs on the unified notebook's `TRADE_DATE` (same data as the SANOS EKF above) so the comparison is apples-to-apples; only the **hyperparameters** match the gate's eval notebook.

In [13]:
df_tickers = pl.read_csv(f"spy_options_tickers_{TRADE_DATE}.csv")

df_spy = client.fetch_trades("SPY", TRADE_DATE)

In [14]:
df_quotes, df_trades = build_iv_observations(df_tickers, TRADE_DATE, df_spy, min_trades=100)

Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105P00462500: 17 bad quotes
Expiry 2024-01-05: implied forward from 13 strike(s)


In [15]:
# ===== Stage 1: batch DLV fit (same pipeline & config as sanos_dlv_kf_eval_lean.ipynb) =====
from functools import partial

from dlv_fit_absolute import fit_density
from dlv_grid_absolute import make_density0, make_strike_grid
from dlv_pricing_absolute import option_prices
from dlv_snapshot_absolute import make_batch_snapshots, run_intraday_loop

# --- DLV grid / fit config (lean-notebook values) ---
BATCH_FREQ = "1s"  # "1s" | "1m" | "5m"  (1s = faithful to the gate eval; coarser = much faster)
LOWER_SCALE, UPPER_SCALE, N_PAD = 0.9, 1.1, 1
DLV_MIN, DLV_MAX = 0.001, 2.0
CALLS_ONLY, PUTS_ONLY = False, True
REGULARIZER_WEIGHT = 0.0
LOG_DLV_SPACE = False
TEMPORAL_REGULARIZER_WEIGHT = 0.0
TEMPORAL_WEIGHT_CLIP = float("inf")
BID_ASK_PENALTY_WEIGHT = 0.0

# Normalise prices by discount factor (absolute-strike convention: C/df, P/df)
_has_df = "discount_factor" in df_quotes.columns
_df_scale = pl.col("discount_factor") if _has_df else pl.lit(1.0)
df_q = df_quotes.drop_nulls().with_columns(
    (pl.col("bid_price") / _df_scale).alias("bid_norm"),
    (pl.col("ask_price") / _df_scale).alias("ask_norm"),
    ((pl.col("bid_price") + pl.col("ask_price")) / 2.0 / _df_scale).alias("mid_norm"),
    pl.col("strike").alias("pure_strike"),
)
df_calls = df_q.filter(pl.col("is_call"))
df_puts = df_q.filter(~pl.col("is_call"))
tte_y_max = float(df_q["T"].max())
tte_y_med = float(df_q["T"].median())

# Strike grid + initial density
tte_geom = tte_y_max * 1.1
df_anchor = df_puts if PUTS_ONLY else df_calls
ref_fwd = float(df_anchor["forward"].median())
obs_ks = np.sort(df_anchor["strike"].unique().to_numpy().astype(float))
grid = make_strike_grid(
    obs_ks, fwd=ref_fwd, lower_scale=LOWER_SCALE, upper_scale=UPPER_SCALE, n_pad=N_PAD
)
xstrikes_row = grid["xstrikes_row"]
anchor_pure = grid["anchor_pure"]
grid_K, density0 = make_density0(xstrikes_row, fwd=ref_fwd)
dlv_anchor_pure = anchor_pure[1:-1]  # (N-2,) strikes carrying the DLV² state
N_dlv = len(dlv_anchor_pure)

_fit_fn = partial(
    fit_density,
    xstrikes_row=xstrikes_row,
    grid_K=grid_K,
    density0=density0,
    tte_geom=tte_geom,
    dlv_min=DLV_MIN,
    dlv_max=DLV_MAX,
    regularizer_weight=REGULARIZER_WEIGHT,
    log_dlv_space=LOG_DLV_SPACE,
    temporal_regularizer_weight=TEMPORAL_REGULARIZER_WEIGHT,
    temporal_weight_clip=TEMPORAL_WEIGHT_CLIP,
    bid_ask_penalty_weight=BID_ASK_PENALTY_WEIGHT,
)

# Batch snapshots + intraday DLV fit
snaps_calls = make_batch_snapshots(df_calls, BATCH_FREQ)
snaps_puts = make_batch_snapshots(df_puts, BATCH_FREQ)
snap_ref = snaps_puts if PUTS_ONLY else snaps_calls
batch_ts_list = sorted(snap_ref["batch_ts"].unique().to_list())

try:
    from tqdm.auto import tqdm
except ImportError:

    def tqdm(x, **kw):
        return x


results = run_intraday_loop(
    batch_ts_list=tqdm(batch_ts_list, desc="Fitting DLV snapshots"),
    snaps_calls=snaps_calls,
    snaps_puts=snaps_puts,
    fit_fn=_fit_fn,
    tte_y_med=tte_y_med,
    calls_only=CALLS_ONLY,
    puts_only=PUTS_ONLY,
    verbose=False,
    anchor_pure=anchor_pure,
)
n_failed = sum(r["failed"] for r in results)
print(f"Stage 1 done — {len(results):,} snapshots ({N_dlv} DLV² dims), {n_failed} failed")

Stage 1 done — 24,066 snapshots (21 DLV² dims), 0 failed


In [25]:
# ===== Stage 2: correlated-Q Kalman filter on DLV² (lean-notebook KF_* hyperparameters) =====
from dlv_kalman_filter import KFConfig, run_dlv_kalman_filter_v2

# Kalman filter hyperparameters — identical to sanos_dlv_kf_eval_lean.ipynb
KF_Q_SCALE = 1e5  # process noise variance per trading day (per DLV² component)
KF_R_DIAG = 1e-15  # observation noise variance for fresh nodes (r_fresh)
KF_R_STALE = 1e-15  # observation noise for stale nodes; inf = selection-H
KF_P0_SCALE = 1e-6  # initial posterior covariance diagonal
TRADING_DAY_S = 23400.0  # 6.5 h in seconds
KF_IGNORE_AT_DLV_MAX = True
KF_ELL = 3.0  # RBF correlation length (dollars)
KF_STENCIL = 3  # 1 = permissive, 3 = conservative fresh-node rule
KF_P0_CORRELATED = True
KF_SOFT_FLOOR = 0.0

cfg_v2 = KFConfig(
    q_scale=KF_Q_SCALE,
    ell=KF_ELL,
    r_diag=KF_R_DIAG,
    r_stale=KF_R_STALE,
    p0_scale=KF_P0_SCALE,
    p0_correlated=KF_P0_CORRELATED,
    stencil=KF_STENCIL,
    soft_floor=KF_SOFT_FLOOR,
    trading_day_s=TRADING_DAY_S,
    ignore_at_dlv_max=KF_IGNORE_AT_DLV_MAX,
    dlv_max_sq_net=DLV_MAX**2 - DLV_MIN**2,
)
x_hist, x_pred_hist, P_diag_hist, innov_hist, ts_hist, failed_hist, n_fresh_hist = (
    run_dlv_kalman_filter_v2(results, N_dlv=N_dlv, grid_K_nodes=dlv_anchor_pure, cfg=cfg_v2)
)
print(f"Stage 2 done — KF over {len(results):,} steps, {int(failed_hist.sum())} failed")


# ── KF DLV² state → density (N×N exactly-determined system) → standard eval table ──
_kf_N = len(grid_K)
_kf_box = grid_K**2 / (xstrikes_row[2:] - xstrikes_row[:-2]) * tte_geom
_kf_box[0] = 0.0
_kf_box[-1] = 0.0
_kf_pay = np.maximum(grid_K[np.newaxis, :] - grid_K[:, np.newaxis], 0.0)  # pay[j,i]=max(K_i-K_j,0)


def _density_from_kf_dlvsq(dlvsq_net, fwd, dlv_min=DLV_MIN):
    """Reconstruct density from (N-2,) net-DLV²; None if solve fails or >2% neg. mass."""
    N = _kf_N
    v = dlvsq_net + dlv_min**2
    A = np.zeros((N, N))
    b = np.zeros(N)
    A[0] = 1.0
    b[0] = 1.0  # sum d = 1
    A[1] = grid_K
    b[1] = fwd  # K @ d = fwd
    for j in range(1, N - 1):
        A[j + 1] = _kf_pay[j]
        A[j + 1, j] -= v[j - 1] * _kf_box[j]
        b[j + 1] = max(fwd - grid_K[j], 0.0)
    try:
        d = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None
    if (-np.minimum(d, 0.0)).sum() > 0.02:
        return None
    d = np.maximum(d, 0.0)
    s = d.sum()
    return d / s if s > 1e-12 else None


# prior = x_pred_hist[k] (held-out, pre-update); post = x_hist[k] (in-sample, post-update)
_snap_src = snaps_puts if PUTS_ONLY else snaps_calls
_is_call_flag = not PUTS_ONLY
_rec = {k: [] for k in ("ts", "strike", "forward", "is_call", "bid", "ask", "prior", "post")}

for k, r in enumerate(results):
    if r["failed"]:
        continue
    snap = _snap_src.filter(pl.col("batch_ts") == r["ts"]).sort("pure_strike")
    if snap.is_empty():
        continue
    strikes_k = snap["pure_strike"].to_numpy().astype(float)
    bid_k = snap["bid_norm"].to_numpy().astype(float)
    ask_k = snap["ask_norm"].to_numpy().astype(float)
    fwd_k = r["fwd"]
    is_call_k = np.full(len(strikes_k), _is_call_flag, dtype=bool)

    d_post = _density_from_kf_dlvsq(x_hist[k], fwd_k)
    d_prior = _density_from_kf_dlvsq(x_pred_hist[k], fwd_k)
    if d_post is None or d_prior is None:
        continue
    px_post = option_prices(d_post, grid_K, strikes_k, is_call_k, fwd_k)
    px_prior = option_prices(d_prior, grid_K, strikes_k, is_call_k, fwd_k)

    n = len(strikes_k)
    _rec["ts"].extend([k] * n)
    _rec["strike"].extend(strikes_k)
    _rec["forward"].extend([fwd_k] * n)
    _rec["is_call"].extend(is_call_k)
    _rec["bid"].extend(bid_k)
    _rec["ask"].extend(ask_k)
    _rec["prior"].extend(px_prior)
    _rec["post"].extend(px_post)

# bid_norm/ask_norm and model prices are already undiscounted (C/df, P/df) → discount_factor=1
df_gate = um.from_option_prices(
    ts=_rec["ts"],
    strike=_rec["strike"],
    forward=_rec["forward"],
    is_call=_rec["is_call"],
    bid=_rec["bid"],
    ask=_rec["ask"],
    prior_price=_rec["prior"],
    post_price=_rec["post"],
)
m_gate = um.evaluate(df_gate, model_name="SANOS 2-stage (DLV fit + KF)")
rows.append(m_gate)
m_gate

Stage 2 done — KF over 24,066 steps, 0 failed


{'model': 'SANOS 2-stage (DLV fit + KF)',
 'ho_nse_mean': 1.7720965917244371,
 'ho_nse_med': 0.6673083856861617,
 'is_nse_mean': 0.4450232615414862,
 'is_nse_med': 0.00023621197586022857,
 'ho_rmse_mean': 0.022639266437859906,
 'ho_rmse_med': 0.011004715611919718,
 'is_rmse_mean': 0.0052930818701166825,
 'is_rmse_med': 2.0926331442141545e-06,
 'neg_mass_mean': 0.0,
 'neg_mass_med': 0.0,
 'arb_frac': 0.0,
 'n_snapshots': 24066}

## 3. KF on splines / parameters

`run_spline_surface_kalman` now emits both `kalman_price` (post-update) and `kalman_price_prior` (pre-update / held-out). `from_option_prices` converts raw bid/ask + model prices into the standard call-price table.

Negative mass: reconstruct the smile across a common strike grid at sampled times (apply `cfg.model.observe(theta, k, T)` over the grid), price with `bs_call_price`, then `negative_mass_surface`.

In [27]:
import surface_model as sm

# df_quotes_iv / df_trades_iv are the IV-annotated outputs of build_iv_observations
# (already computed above as df_quotes / df_trades — just alias them here)
df_quotes_iv = df_quotes
df_trades_iv = df_trades

# Spline surface KF config
# SVI_SLICE_MODEL: 5-param Gatheral SVI, single-expiry (good for short-dated SPY)
# Swap for sm.QUAD_SLICE_MODEL (3 params) or sm.BSPLINE_SLICE_MODEL (8 params) to compare
cfg = sm.SurfaceModelConfig(
    model=sm.SVI_SLICE_MODEL,
    Q_scale=1e-4,  # process noise per quote (tune: higher = faster-tracking smile)
    Q_structure="diagonal",  # per-param scaling via model.q_diag_weights
    P0_scale=1.0,  # initial uncertainty (large → flat start, KF converges quickly)
)

In [ ]:
from surface_kalman_filters import run_spline_surface_kalman

q_res, t_res, df_state = run_spline_surface_kalman(df_quotes_iv, df_trades_iv, cfg)
q = q_res.to_pandas()
df_spline = um.from_option_prices(
    ts=q["ts_ns"],
    strike=q["strike"],
    forward=q["forward"],
    is_call=q["is_call"],
    bid=q["bid_price"],
    ask=q["ask_price"],
    prior_price=q["kalman_price_prior"],
    post_price=q["kalman_price"],
    discount_factor=q["discount_factor"] if "discount_factor" in q else None,
)

In [30]:
# ── Negative mass: reconstruct smile over a strike grid at sampled snapshot times ──
state_pd = df_state.to_pandas().sort_values("ts_ns").reset_index(drop=True)
n_params = cfg.n_params
theta_cols = [f"theta_{j}" for j in range(n_params)]

# Strike grid spanning the observed strikes
K_lo = float(df_quotes_iv["strike"].min())
K_hi = float(df_quotes_iv["strike"].max())
grid_K = np.linspace(K_lo, K_hi, 41)

# Subsample snapshot times for speed (1 000 is usually plenty)
n_snap = min(1000, len(state_pd))
samp_idx = np.linspace(0, len(state_pd) - 1, n_snap, dtype=int)
state_samp = state_pd.iloc[samp_idx].reset_index(drop=True)

# For each sampled ts: look up the most-recent forward/T from the quote stream
q_sorted = q.sort_values("ts_ns").reset_index(drop=True)
asof = np.searchsorted(q_sorted["ts_ns"].values, state_samp["ts_ns"].values, side="right") - 1
asof = np.clip(asof, 0, len(q_sorted) - 1)
F_vec = q_sorted["forward"].values[asof]
T_vec = q_sorted["T"].values[asof]
thetas_samp = state_samp[theta_cols].values  # (n_snap, n_params)

# Reconstruct call-price matrix (n_snap, n_grid)
C_mat = np.full((n_snap, len(grid_K)), np.nan)
for i in range(n_snap):
    theta_i, F_i, T_i = thetas_samp[i], F_vec[i], T_vec[i]
    if not (np.isfinite(theta_i).all() and F_i > 0 and np.isfinite(F_i) and T_i > 0):
        continue
    ivs = np.array([cfg.model.observe(theta_i, np.log(k / F_i), T_i)[0] for k in grid_K])
    C_mat[i] = um.bs_call_price(np.full(len(grid_K), F_i), grid_K, np.full(len(grid_K), T_i), ivs)

valid = np.isfinite(C_mat).all(axis=1)
nm = um.negative_mass_surface(grid_K, C_mat[valid], F_vec[valid])

m_spline = um.evaluate(df_spline, model_name="KF spline", neg_mass_values=nm)
rows.append(m_spline)
m_spline

{'model': 'KF spline',
 'ho_nse_mean': 6.442244375247699,
 'ho_nse_med': 1.7881351997150705,
 'is_nse_mean': 1.008814395077012,
 'is_nse_med': 0.22140865379523028,
 'ho_rmse_mean': 0.08998693027212974,
 'ho_rmse_med': 0.04364873557140925,
 'is_rmse_mean': 0.020773265666934024,
 'is_rmse_med': 0.004750950859978766,
 'neg_mass_mean': 0.0,
 'neg_mass_med': 0.0,
 'arb_frac': 0.0,
 'n_snapshots': 388}

## 4. KF on single strikes

Random-walk EKF (`f(x)=x`), so the one-step-ahead **prior == previous posterior**: `prior_price` is `kalman_price` shifted one quote back *within each strike*. `build_iv_surface` returns a per-strike `kalman_price`.

In [38]:
from personal.shared_functions.kalman_filters import build_iv_surface

df_quotes_iv, df_trades_iv = build_iv_surface(
    df_tickers, TRADE_DATE, df_spy=df_spy, min_trades=100, simple=False
)


q = df_quotes_iv.sort(["strike", "is_call", "ts_ns"]).to_pandas()
# one-step-ahead prior = previous posterior within each (strike, is_call)
q["kalman_price_prior"] = q.groupby(["strike", "is_call"])["kalman_price"].shift(1)
q = q.dropna(subset=["kalman_price_prior"])
df_single = um.from_option_prices(
    ts=q["ts_ns"],
    strike=q["strike"],
    forward=q["forward"],
    is_call=q["is_call"],
    bid=q["bid_price"],
    ask=q["ask_price"],
    prior_price=q["kalman_price_prior"],
    post_price=q["kalman_price"],
    discount_factor=q["discount_factor"] if "discount_factor" in q else None,
)

O:SPY240105C00469000: dropping 2 bad quotes (crossed×2 (first=2024-01-03 11:49:45.633413-05:00))
O:SPY240105C00472000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 15:51:02.275318-05:00))
O:SPY240105C00473000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:29:17.744517-05:00))
O:SPY240105C00474000: dropping 4 bad quotes (crossed×4 (first=2024-01-03 10:25:56.925706-05:00))
O:SPY240105C00475000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 14:02:06.496803-05:00))
O:SPY240105C00477000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:00:44.569057-05:00))
O:SPY240105C00481000: dropping 46 bad quotes (sub-tick bid×46 (first=2024-01-03 16:00:02.443177-05:00))
O:SPY240105C00482000: dropping 64 bad quotes (sub-tick bid×64 (first=2024-01-03 10:00:01.421753-05:00))
O:SPY240105C00483000: dropping 842 bad quotes (sub-tick bid×842 (first=2024-01-03 09:30:00.060278-05:00), zero bid size×1 (first=2024-01-03 09:30:00.060278-05:00))
O:SPY240105C00484000: dropping 2129 bad quotes

In [46]:
# ── Negative mass for single-strike KF ──
# Use post_call from df_single directly (already undiscounted call prices via put-call parity)
# Keep only puts to avoid two quote-streams per strike
puts_mask = ~q["is_call"]  # q still has is_call; filter ts values that came from puts
ds = df_single[df_single["ts"].isin(q.loc[puts_mask, "ts_ns"])].copy()

unique_K_single = np.sort(ds["strike"].unique())

n_snap_single = min(500, ds["ts"].nunique())
ts_int = ds["ts"].astype(np.int64).values
snap_ts_int = np.linspace(ts_int.min(), ts_int.max(), n_snap_single, dtype=np.int64)

strike_groups_s = {
    K: grp.sort_values("ts").reset_index(drop=True) for K, grp in ds.groupby("strike")
}

# Forward from reference strike
ref_K_s = unique_K_single[len(unique_K_single) // 2]
ref_grp_s = strike_groups_s[ref_K_s]
ref_ts_int = ref_grp_s["ts"].astype(np.int64).values
ref_asof_s = np.clip(
    np.searchsorted(ref_ts_int, snap_ts_int, side="right") - 1, 0, len(ref_grp_s) - 1
)
F_arr_s = ref_grp_s["forward"].values[ref_asof_s]

C_mat_s = np.full((n_snap_single, len(unique_K_single)), np.nan)
for j, K_j in enumerate(unique_K_single):
    grp = strike_groups_s[K_j]
    grp_ts_int = grp["ts"].astype(np.int64).values
    idx = np.clip(np.searchsorted(grp_ts_int, snap_ts_int, side="right") - 1, 0, len(grp) - 1)
    C_mat_s[:, j] = grp["post_call"].values[idx]

valid_s = np.isfinite(C_mat_s).all(axis=1) & np.isfinite(F_arr_s)
nm_single = um.negative_mass_surface(unique_K_single, C_mat_s[valid_s], F_arr_s[valid_s])

In [48]:
# negative mass: at sampled times, gather kalman_iv across all strikes (asof / pivot),
# price with um.bs_call_price -> C_mat, then um.negative_mass_surface(grid_K, C_mat, F_vec)
m_single = um.evaluate(df_single, model_name="KF single-strike", neg_mass_values=nm_single)
rows.append(m_single)
m_single

{'model': 'KF single-strike',
 'ho_nse_mean': 0.4080056912026887,
 'ho_nse_med': 0.28502434135235244,
 'is_nse_mean': 0.34764174647198776,
 'is_nse_med': 0.27134071922569025,
 'ho_rmse_mean': 0.007105354828325362,
 'ho_rmse_med': 0.0031195201530067607,
 'is_rmse_mean': 0.006343210689758206,
 'is_rmse_med': 0.00277563240155132,
 'neg_mass_mean': -0.7042635307281694,
 'neg_mass_med': -0.6732671260796516,
 'arb_frac': 1.0,
 'n_snapshots': 2133013}

## Comparison table

In [49]:
um.comparison_table(rows)

,ho_nse_mean,ho_nse_med,is_nse_mean,is_nse_med,ho_rmse_mean,ho_rmse_med,is_rmse_mean,is_rmse_med,neg_mass_mean,neg_mass_med,arb_frac,n_snapshots
model,,,,,,,,,,,,
SANOS EKF,0.470019,0.423749,0.193867,1.827297e-01,0.008731,0.007142,0.005225,3.941555e-03,0.000000e+00,0.000000,0.0,24066
SANOS gate,10.371440,5.888930,137.036315,6.071472e+01,0.110523,0.086883,1.063827,7.480035e-01,0.000000e+00,0.000000,0.0,2075773
SANOS gate,17.531849,15.000000,128.393981,1.155495e+02,0.206185,0.075000,2.339759,1.485465e+00,0.000000e+00,0.000000,0.0,100648
SANOS 2-stage (DLV fit + KF),39.950944,28.481756,37.182044,2.612261e+01,0.337719,0.255261,0.305382,2.296995e-01,-4.738323e-16,0.000000,0.0,24066
SANOS 2-stage (DLV fit + KF),12.684891,4.548299,9.571313,3.426163e+00,0.116158,0.042491,0.082447,2.902050e-02,-5.178030e-17,0.000000,0.0,24066
SANOS 2-stage (DLV fit + KF),6.147091,1.325437,2.290912,3.258990e-02,0.064552,0.017772,0.021857,1.489642e-03,-1.107717e-16,0.000000,0.0,24066
SANOS 2-stage (DLV fit + KF),4.201671,1.085391,3.069633,2.901687e-01,0.043535,0.014913,0.029671,4.128040e-03,-1.191549e-16,0.000000,0.0,24066
SANOS 2-stage (DLV fit + KF),4.164961,1.080234,3.050818,2.953428e-01,0.043418,0.014982,0.029450,4.076061e-03,-1.094241e-16,0.000000,0.0,24066
SANOS 2-stage (DLV fit + KF),1.772002,0.667142,0.444616,6.586915e-10,0.022639,0.011006,0.005291,5.211198e-11,0.000000e+00,0.000000,0.0,24066
